In [1]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your new DBpedia test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "final_model_mrr_0.4487.pt")

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")

/upb/users/m/manzoor/profiles/unix/cs/.conda/envs/conda-jupyterhub/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Success! Project root set to: /upb/users/m/manzoor/profiles/unix/cs/SPARQL-Embeddings-main-refactored
Dataset Loaded -> Test Split: 337
Evaluation Ready -> Anchors: 337 | Corpus: 1677


In [2]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen
evaluate_sentence_transformer('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/final_model_mrr_0.4487.pt...

SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1316  0.0411  0.0574  0.1716  0.2285    0.1316  0.1306
@5     0.1288  0.0660  0.0789  0.1845  0.2849    0.1288  0.1327
@10    0.1246  0.1226  0.1107  0.2015  0.4065    0.1246  0.1444
@20    0.1150  0.2294  0.1384  0.2116  0.5490    0.1150  0.1786

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0445  0.0213  0.0270  0.0628  0.1335    0.0445  0.0381
@5     0.0362  0.0233  0.0251  0.0725  0.1721    0.0362  0.0362
@10    0.0217  0.0255  0.0200  0.0732  0.1780    0.0217  0.0307
@20    0.0154  0.0305  0.0177  0.0751  0.2107    0.0154  0.0318


No sentence-transformers model found with name Qwen/Qwen2.5-0.5B-Instruct. Creating a new one with mean pooling.



E5-base-v2
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0297  0.0134  0.0162  0.0554  0.0831    0.0297  0.0309
@5     0.0237  0.0176  0.0170  0.0587  0.0979    0.0237  0.0290
@10    0.0208  0.0303  0.0205  0.0633  0.1306    0.0208  0.0321
@20    0.0209  0.0527  0.0251  0.0682  0.2077    0.0209  0.0409

Qwen2.5-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0386  0.0157  0.0195  0.0668  0.0801    0.0386  0.0428
@5     0.0350  0.0211  0.0228  0.0709  0.0979    0.0350  0.0419
@10    0.0320  0.0327  0.0280  0.0747  0.1246    0.0320  0.0429
@20    0.0273  0.0492  0.0310  0.0784  0.1810    0.0273  0.0480


In [2]:

display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10482.45it/s]



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0218  0.0099  0.0121  0.0425  0.0653    0.0218  0.0227
@5     0.0184  0.0130  0.0131  0.0454  0.0772    0.0184  0.0224
@10    0.0196  0.0246  0.0182  0.0517  0.1276    0.0196  0.0268
@20    0.0191  0.0447  0.0230  0.0560  0.1899    0.0191  0.0349


In [4]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8534.77it/s]



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0386  0.0143  0.0182  0.0623  0.0861    0.0386  0.0392
@5     0.0344  0.0228  0.0231  0.0669  0.1068    0.0344  0.0390
@10    0.0297  0.0357  0.0269  0.0717  0.1424    0.0297  0.0403
@20    0.0303  0.0647  0.0355  0.0788  0.2463    0.0303  0.0518
